# Mini-Project 2 — Part 4: Evaluation, Visualization & Ablation
**Metrics | Visualizations | Ablation Results**

## 1. Setup & Load Models

In [ ]:
!pip install torch torchvision torchmetrics scipy scikit-learn matplotlib -q

In [ ]:
import json, os, pickle
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import distance_transform_edt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from torchvision import transforms
from torchvision.datasets import VOCSegmentation
from torch.utils.data import DataLoader

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = '/content/drive/MyDrive/mini_proj2_checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

with open('/content/project_config.json') as f:
    cfg = json.load(f)
VOC_ROOT    = cfg['voc_root']
IMG_SIZE    = tuple(cfg['img_size'])
NUM_CLASSES = cfg['num_classes']
VOC_CLASSES = cfg['voc_classes']

VOC_COLORMAP = np.array([
    [0,0,0],[128,0,0],[0,128,0],[128,128,0],[0,0,128],
    [128,0,128],[0,128,128],[128,128,128],[64,0,0],[192,0,0],
    [64,128,0],[192,128,0],[64,0,128],[192,0,128],[64,128,128],
    [192,128,128],[0,64,0],[128,64,0],[0,192,0],[128,192,0],[0,64,128]
], dtype=np.uint8)

## 2. Load U-Net & Run Inference

In [ ]:
# Rebuild the same U-Net class from 02_unet.ipynb
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=21, features=[64, 128, 256, 512]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups   = nn.ModuleList()
        self.pool  = nn.MaxPool2d(2, 2)
        ch = in_channels
        for f in features:
            self.downs.append(DoubleConv(ch, f)); ch = f
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(f * 2, f, 2, 2))
            self.ups.append(DoubleConv(f * 2, f))
        self.final = nn.Conv2d(features[0], num_classes, 1)
    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x); skips.append(x); x = self.pool(x)
        x = self.bottleneck(x); skips = skips[::-1]
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip = skips[i // 2]
            if x.shape != skip.shape:
                x = nn.functional.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1); x = self.ups[i + 1](x)
        return self.final(x)

unet = UNet(num_classes=NUM_CLASSES).to(DEVICE)
ckpt = torch.load(os.path.join(SAVE_DIR, 'unet_augTrue_lossce_best.pth'), map_location=DEVICE)
unet.load_state_dict(ckpt['model_state'])
unet.eval()
print(f'U-Net loaded (epoch {ckpt["epoch"]}, mIoU={ckpt["miou"]:.4f})')

In [ ]:
transform_img = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
transform_mask = transforms.Compose([
    transforms.Resize(IMG_SIZE, interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(),
])
val_set    = VOCSegmentation(VOC_ROOT, '2007', 'val', download=False,
                             transform=transform_img, target_transform=transform_mask)
val_loader = DataLoader(val_set, batch_size=8, shuffle=False, num_workers=2)

unet_preds, unet_gts = [], []
with torch.no_grad():
    for imgs, masks in val_loader:
        imgs = imgs.to(DEVICE)
        preds = unet(imgs).argmax(dim=1).cpu()
        unet_preds.append(preds)
        unet_gts.append(masks.squeeze(1))

unet_preds = torch.cat(unet_preds).numpy()
unet_gts   = torch.cat(unet_gts).numpy()
print(f'U-Net inference done: {len(unet_preds)} images')

## 3. Evaluation Metrics

In [ ]:
def compute_metrics(preds, gts, num_classes, ignore=255):
    """Returns: pixel_acc, miou, per_class_iou, per_class_acc"""
    total_tp = total_pixels = 0
    per_iou  = np.zeros(num_classes)
    per_acc  = np.zeros(num_classes)
    per_cnt  = np.zeros(num_classes)

    for p, g in zip(preds, gts):
        mask = g != ignore
        total_tp     += (p[mask] == g[mask]).sum()
        total_pixels += mask.sum()
        for c in range(num_classes):
            tp  = ((p == c) & (g == c) & mask).sum()
            fp  = ((p == c) & (g != c) & mask).sum()
            fn  = ((p != c) & (g == c) & mask).sum()
            denom = tp + fp + fn
            if denom > 0:
                per_iou[c] += tp / denom
                per_cnt[c] += 1
            gt_c = (g == c) & mask
            if gt_c.sum() > 0:
                per_acc[c] += ((p == c) & gt_c).sum() / gt_c.sum()

    valid = per_cnt > 0
    per_iou[valid] /= per_cnt[valid]
    per_acc[valid] /= per_cnt[valid]

    return {
        'pixel_acc':    total_tp / total_pixels,
        'miou':         per_iou[valid].mean(),
        'per_class_iou': per_iou,
        'per_class_acc': per_acc,
    }

unet_metrics = compute_metrics(unet_preds, unet_gts, NUM_CLASSES)
print(f"\nU-Net Results")
print(f"  Pixel Accuracy: {unet_metrics['pixel_acc']:.4f}")
print(f"  mIoU:           {unet_metrics['miou']:.4f}")

In [ ]:
def hd95(pred, gt, class_id, ignore=255):
    """95th percentile Hausdorff Distance for a single class."""
    mask = gt != ignore
    pred_b = (pred == class_id) & mask
    gt_b   = (gt   == class_id) & mask
    if not pred_b.any() or not gt_b.any():
        return np.nan
    dist_pred = distance_transform_edt(~pred_b)
    dist_gt   = distance_transform_edt(~gt_b)
    d1 = dist_gt[pred_b]
    d2 = dist_pred[gt_b]
    return np.percentile(np.concatenate([d1, d2]), 95)

# Compute HD95 per class (slow — uses subset)
N_HD = 50
hd95_scores = {}
for c in range(1, NUM_CLASSES):  # skip background
    vals = [hd95(unet_preds[i], unet_gts[i], c) for i in range(N_HD)]
    vals = [v for v in vals if not np.isnan(v)]
    hd95_scores[VOC_CLASSES[c]] = np.mean(vals) if vals else np.nan

print('\nHD95 per class (U-Net):')
for name, val in hd95_scores.items():
    print(f'  {name:15s}: {val:.2f}' if not np.isnan(val) else f'  {name:15s}: N/A')

## 4. Per-Class IoU Bar Chart

In [ ]:
plt.figure(figsize=(14, 5))
plt.bar(VOC_CLASSES, unet_metrics['per_class_iou'], color='steelblue')
plt.axhline(unet_metrics['miou'], color='red', linestyle='--', label=f"mIoU={unet_metrics['miou']:.3f}")
plt.xticks(rotation=45, ha='right')
plt.ylabel('IoU'); plt.title('U-Net: Per-Class IoU'); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'unet_per_class_iou.png'), dpi=150)
plt.show()

## 5. Qualitative Visualizations — Best & Worst Predictions

In [ ]:
# Per-image mIoU (focus on 'person' class as suggested by assignment)
MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

def per_image_iou(pred, gt, class_id=None, num_classes=21, ignore=255):
    mask = gt != ignore
    if class_id is not None:
        tp = ((pred == class_id) & (gt == class_id) & mask).sum()
        fp = ((pred == class_id) & (gt != class_id) & mask).sum()
        fn = ((pred != class_id) & (gt == class_id) & mask).sum()
        denom = tp + fp + fn
        return tp / denom if denom > 0 else np.nan
    ious = []
    for c in range(num_classes):
        tp = ((pred == c) & (gt == c) & mask).sum()
        fp = ((pred == c) & (gt != c) & mask).sum()
        fn = ((pred != c) & (gt == c) & mask).sum()
        d  = tp + fp + fn
        if d > 0: ious.append(tp/d)
    return np.mean(ious) if ious else np.nan

scores = [per_image_iou(unet_preds[i], unet_gts[i]) for i in range(len(unet_preds))]
valid  = [(s, i) for i, s in enumerate(scores) if not np.isnan(s)]
valid.sort(key=lambda x: x[0])
worst3 = [i for _, i in valid[:3]]
best3  = [i for _, i in valid[-3:]]

def show_mosaic(indices, title, dataset):
    fig, axes = plt.subplots(len(indices), 3, figsize=(12, 4*len(indices)))
    for row, idx in enumerate(indices):
        img_t, _ = dataset[idx]
        img_np = np.clip(img_t.permute(1,2,0).numpy() * STD + MEAN, 0, 1)
        gt_c   = VOC_COLORMAP[np.clip(unet_gts[idx],   0, 20)]
        pr_c   = VOC_COLORMAP[np.clip(unet_preds[idx], 0, 20)]
        axes[row,0].imshow(img_np); axes[row,0].set_title(f'Image [{idx}]')
        axes[row,1].imshow(gt_c);   axes[row,1].set_title('Ground Truth')
        axes[row,2].imshow(pr_c);   axes[row,2].set_title(f'Pred (mIoU={scores[idx]:.3f})')
        for ax in axes[row]: ax.axis('off')
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f'{title.replace(" ","_")}.png'), dpi=150)
    plt.show()

show_mosaic(best3,  'Top 3 Best Predictions',  val_set)
show_mosaic(worst3, 'Top 3 Worst Predictions', val_set)

## 6. U-Net vs SAM2 Comparison

In [ ]:
# Load SAM2 results saved from 03_sam2.ipynb
with open(os.path.join(SAVE_DIR, 'sam2_preds.pkl'), 'rb') as f:
    sam2_data = pickle.load(f)

sam2_preds = sam2_data['preds']
sam2_gts   = sam2_data['gts']
n          = sam2_data['n_eval']

sam2_metrics = compute_metrics(
    np.array(sam2_preds), np.array(sam2_gts), NUM_CLASSES
)
unet_metrics_sub = compute_metrics(
    unet_preds[:n], unet_gts[:n], NUM_CLASSES
)

print(f'\n=== Comparison (first {n} test images) ===')
print(f'{"Model":<20} {"Pixel Acc":>12} {"mIoU":>10}')
print('-'*44)
print(f'{"U-Net":<20} {unet_metrics_sub["pixel_acc"]:>12.4f} {unet_metrics_sub["miou"]:>10.4f}')
print(f'{"SAM2 (zero-shot)":<20} {sam2_metrics["pixel_acc"]:>12.4f} {sam2_metrics["miou"]:>10.4f}')

## 7. Ablation Study Results

In [ ]:
# Fill in after running 02_unet.ipynb with different configurations
ablation = [
    {'Variant': 'U-Net, CE Loss, No Augmentation',          'mIoU': None, 'Pixel Acc': None},
    {'Variant': 'U-Net, CE Loss, With Augmentation',        'mIoU': None, 'Pixel Acc': None},
    {'Variant': 'U-Net, Dice Loss, With Augmentation',      'mIoU': None, 'Pixel Acc': None},
    {'Variant': 'SAM2 Zero-shot (bbox prompts)',             'mIoU': sam2_metrics['miou'], 'Pixel Acc': sam2_metrics['pixel_acc']},
]

print(f'{"Variant":<45} {"mIoU":>8} {"Pixel Acc":>12}')
print('-' * 68)
for r in ablation:
    miou = f"{r['mIoU']:.4f}" if r['mIoU'] is not None else 'TBD'
    acc  = f"{r['Pixel Acc']:.4f}" if r['Pixel Acc'] is not None else 'TBD'
    print(f"{r['Variant']:<45} {miou:>8} {acc:>12}")

## 8. Confusion Matrix (Optional)

In [ ]:
flat_preds = unet_preds.flatten()
flat_gts   = unet_gts.flatten()
valid_mask = flat_gts != 255
cm = confusion_matrix(flat_gts[valid_mask], flat_preds[valid_mask],
                      labels=list(range(NUM_CLASSES)), normalize='true')

fig, ax = plt.subplots(figsize=(14, 12))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=VOC_CLASSES)
disp.plot(ax=ax, colorbar=True, xticks_rotation=45, values_format='.2f')
plt.title('U-Net Confusion Matrix (normalised)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'unet_confusion_matrix.png'), dpi=150)
plt.show()